In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

In [2]:
# device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
# Define Hyperparameters
batch_size = 64
num_workers = 2
learning_rate = 1e-3
epochs = 5

In [4]:
# Transform
## Imagenet stats, because model was trained on imagenet data
imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]
transform_train = transforms.Compose([
    transforms.Resize(224), # renet expects 224x224 images
    transforms.RandomCrop(224, padding=16),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])
transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

In [6]:
#Load data
train_set = datasets.CIFAR10(
    root="./data",
    train=True,
    download=True,
    transform=transform_train
)
test_set = datasets.CIFAR10(
    root="./data",
    train=False,
    download=True,
    transform=transform_test
)

train_loader = DataLoader(
    train_set,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers
)
test_loader = DataLoader(
    test_set,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers
)

In [7]:
classes = ('plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck')

In [8]:
# Load pretrained model and replace final layer
from torchvision.models import resnet50, ResNet50_Weights

model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# free all layer
for param in model.parameters():
    param.requires_grad = False
# replace last final layer
model.fc = nn.Linear(model.fc.in_features, 10)
model = model.to(device)

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 178MB/s]


In [9]:
# loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(
    model.fc.parameters(),
    lr=learning_rate
)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=epochs)

In [10]:
# Training loop
def train_one_epoch():
    model.train()
    running_loss, correct, total = 0,0,0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss+=loss
        _, predictions = outputs.max(1)
        correct += predictions.eq(labels).sum().item()
        total += labels.size(0)
    return running_loss/ len(train_loader), 100. * correct/total

In [11]:
# evaluation loop
def evaluate():
    model.eval()
    correct, total = 0,0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predictions = outputs.max(1)
            correct += predictions.eq(labels).sum().item()
            total += labels.size(0)
    return 100. * correct/total

In [ ]:
# training model
for epoch in range(epochs):
    train_loss, train_acc = train_one_epoch()
    test_acc = evaluate()
    scheduler.step()
    print(f"Epoch {epoch+1}/{epochs} | Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | Test Acc: {test_acc:.2f}%")

Epoch 1/5 | Loss: 0.7596 | Train Acc: 76.59% | Test Acc: 81.91%
